In [1]:
# Q3 - Job and education

# Question: which jobs and education levels have the highest yes rate?

# Each group is compared with the overall yes rate (the Q1 baseline) and shown with its customer count. Groups under 100 customers are flagged and not interpreted. "unknown" is kept as its own group (decision 1).

In [2]:
import pandas as pd

CLEAN_PATH = "../../data/bank_marketing_clean.parquet"
clean_df = pd.read_parquet(CLEAN_PATH)

clean_df.shape

(41176, 22)

In [3]:
# Q3, step 1: yes rate by job
subscribed = clean_df["subscribed_term_deposit"] == "yes"
baseline_rate = subscribed.mean() * 100

job_summary = (
    pd.DataFrame({"job": clean_df["job"], "subscribed": subscribed})
    .groupby("job")["subscribed"]
    .agg(customers="count", subscribers="sum")
)
job_summary["yes_rate_pct"] = (job_summary["subscribers"] / job_summary["customers"] * 100).round(2)
job_summary["vs_baseline_points"] = (job_summary["yes_rate_pct"] - baseline_rate).round(2)
job_summary["share_of_customers_pct"] = (job_summary["customers"] / len(clean_df) * 100).round(2)
job_summary["share_of_subscribers_pct"] = (job_summary["subscribers"] / subscribed.sum() * 100).round(2)
job_summary["small_group"] = job_summary["customers"] < 100

print(f"Baseline yes rate: {baseline_rate:.2f}%")
print(f"Customers in groups: {job_summary['customers'].sum()} (should be {len(clean_df)})")
print(f"Subscribers in groups: {job_summary['subscribers'].sum()} (should be {subscribed.sum()})")
print()
job_summary.sort_values("yes_rate_pct", ascending=False)

Baseline yes rate: 11.27%
Customers in groups: 41176 (should be 41176)
Subscribers in groups: 4639 (should be 4639)



,customers,subscribers,yes_rate_pct,vs_baseline_points,share_of_customers_pct,share_of_subscribers_pct,small_group
job,,,,,,,
student,875,275,31.43,20.16,2.13,5.93,False
retired,1718,434,25.26,13.99,4.17,9.36,False
unemployed,1014,144,14.20,2.93,2.46,3.10,False
admin.,10419,1351,12.97,1.70,25.30,29.12,False
management,2924,328,11.22,-0.05,7.10,7.07,False
unknown,330,37,11.21,-0.06,0.80,0.80,False
technician,6739,730,10.83,-0.44,16.37,15.74,False
self-employed,1421,149,10.49,-0.78,3.45,3.21,False
housemaid,1060,106,10.00,-1.27,2.57,2.28,False


In [4]:
# Q3, step 2: yes rate by education, in education order
education_order = [
    "illiterate", "basic.4y", "basic.6y", "basic.9y",
    "high.school", "professional.course", "university.degree", "unknown",
]

education_summary = (
    pd.DataFrame({"education": clean_df["education"], "subscribed": subscribed})
    .groupby("education")["subscribed"]
    .agg(customers="count", subscribers="sum")
    .reindex(education_order)
)
education_summary["yes_rate_pct"] = (education_summary["subscribers"] / education_summary["customers"] * 100).round(2)
education_summary["vs_baseline_points"] = (education_summary["yes_rate_pct"] - baseline_rate).round(2)
education_summary["share_of_customers_pct"] = (education_summary["customers"] / len(clean_df) * 100).round(2)
education_summary["share_of_subscribers_pct"] = (education_summary["subscribers"] / subscribed.sum() * 100).round(2)
education_summary["small_group"] = education_summary["customers"] < 100

print(f"Baseline yes rate: {baseline_rate:.2f}%")
print(f"Customers in groups: {education_summary['customers'].sum()} (should be {len(clean_df)})")
print(f"Subscribers in groups: {education_summary['subscribers'].sum()} (should be {subscribed.sum()})")
print()
education_summary

Baseline yes rate: 11.27%
Customers in groups: 41176 (should be 41176)
Subscribers in groups: 4639 (should be 4639)



,customers,subscribers,yes_rate_pct,vs_baseline_points,share_of_customers_pct,share_of_subscribers_pct,small_group
education,,,,,,,
illiterate,18,4,22.22,10.95,0.04,0.09,True
basic.4y,4176,428,10.25,-1.02,10.14,9.23,False
basic.6y,2291,188,8.21,-3.06,5.56,4.05,False
basic.9y,6045,473,7.82,-3.45,14.68,10.20,False
high.school,9512,1031,10.84,-0.43,23.10,22.22,False
professional.course,5240,595,11.35,0.08,12.73,12.83,False
university.degree,12164,1669,13.72,2.45,29.54,35.98,False
unknown,1730,251,14.51,3.24,4.20,5.41,False


In [5]:
# Q3, step 3: job against age
age_edges = [0, 25, 35, 45, 55, 65, 120]
age_labels = ["under 25", "25-34", "35-44", "45-54", "55-64", "65 and over"]
age_band = pd.cut(clean_df["age"], bins=age_edges, labels=age_labels, right=False)

job_group = clean_df["job"].where(clean_df["job"].isin(["retired", "student"]), "all other jobs")
group_order = ["retired", "student", "all other jobs"]

table = pd.DataFrame({"age_band": age_band, "job_group": job_group, "subscribed": subscribed})

group_totals = (
    table.groupby("job_group")["subscribed"]
    .agg(customers="count", subscribers="sum")
    .reindex(group_order)
)
group_totals["yes_rate_pct"] = (group_totals["subscribers"] / group_totals["customers"] * 100).round(2)
print("Totals by job group:")
print(group_totals)
print(f"Customers: {group_totals['customers'].sum()} (should be {len(clean_df)}), "
      f"subscribers: {group_totals['subscribers'].sum()} (should be {subscribed.sum()})")
print()

grouped = (
    table.groupby(["age_band", "job_group"], observed=True)["subscribed"]
    .agg(customers="count", subscribers="sum")
)
grouped["yes_rate_pct"] = (grouped["subscribers"] / grouped["customers"] * 100).round(2)

customers_pivot = (grouped["customers"].unstack("job_group")
                   .reindex(columns=group_order).fillna(0).astype(int))
rate_pivot = (grouped["yes_rate_pct"].where(grouped["customers"] >= 100)
              .unstack("job_group").reindex(columns=group_order))

print("Customers by age band and job group:")
print(customers_pivot)
print()
print("Yes rate (%) by age band and job group (blank where under 100 customers):")
print(rate_pivot)
print()

share_in_band = pd.DataFrame({
    "retired_pct_of_band": (customers_pivot["retired"] / customers_pivot.sum(axis=1) * 100).round(1),
    "student_pct_of_band": (customers_pivot["student"] / customers_pivot.sum(axis=1) * 100).round(1),
})
print("Share of each age band that is retired or students:")
print(share_in_band)

Totals by job group:
                customers  subscribers  yes_rate_pct
job_group                                           
retired              1718          434         25.26
student               875          275         31.43
all other jobs      38583         3930         10.19
Customers: 41176 (should be 41176), subscribers: 4639 (should be 4639)

Customers by age band and job group:
job_group    retired  student  all other jobs
age_band                                     
under 25           1      378             688
25-34              8      433           13243
35-44             49       60           13386
45-54            224        4            8474
55-64            877        0            2689
65 and over      559        0             103

Yes rate (%) by age band and job group (blank where under 100 customers):
job_group    retired  student  all other jobs
age_band                                     
under 25         NaN    37.83           16.42
25-34            NaN    

In [6]:
# Q3, step 4: retired vs not retired, ages 50 and over in five-year steps
step_edges = [50, 55, 60, 65, 70, 120]
step_labels = ["50-54", "55-59", "60-64", "65-69", "70 and over"]
age_step = pd.cut(clean_df["age"], bins=step_edges, labels=step_labels, right=False)

retirement_status = clean_df["job"].where(clean_df["job"] == "retired", "not retired")
status_order = ["retired", "not retired"]

older_df = pd.DataFrame({
    "age_step": age_step,
    "retirement_status": retirement_status,
    "subscribed": subscribed,
}).dropna(subset=["age_step"])

grouped = (
    older_df.groupby(["age_step", "retirement_status"], observed=True)["subscribed"]
    .agg(customers="count", subscribers="sum")
)
grouped["yes_rate_pct"] = (grouped["subscribers"] / grouped["customers"] * 100).round(2)

customers_pivot = (grouped["customers"].unstack("retirement_status")
                   .reindex(columns=status_order).fillna(0).astype(int))
subscribers_pivot = (grouped["subscribers"].unstack("retirement_status")
                     .reindex(columns=status_order).fillna(0).astype(int))
rate_pivot = (grouped["yes_rate_pct"].where(grouped["customers"] >= 100)
              .unstack("retirement_status").reindex(columns=status_order))

customers_pivot["total_customers"] = customers_pivot.sum(axis=1)
customers_pivot["retired_pct_of_step"] = (
    customers_pivot["retired"] / customers_pivot["total_customers"] * 100
).round(1)

print("Customers by age step and retirement status:")
print(customers_pivot)
print()
print("Subscribers by age step and retirement status:")
print(subscribers_pivot)
print()
print("Yes rate (%) by age step and retirement status (blank where under 100 customers):")
print(rate_pivot)
print()

print(f"Customers aged 50 and over: {customers_pivot['total_customers'].sum()} (should be 8053)")
print(f"Subscribers aged 50 and over: {subscribers_pivot.values.sum()} (should be 1169)")
print(f"Retired 55-59 + 60-64: {customers_pivot.loc[['55-59', '60-64'], 'retired'].sum()} (should be 877)")
print(f"Retired 65-69 + 70 and over: {customers_pivot.loc[['65-69', '70 and over'], 'retired'].sum()} (should be 559)")
print(f"Retired 50-54: {customers_pivot.loc['50-54', 'retired']} (should be at most 224)")

Customers by age step and retirement status:
retirement_status  retired  not retired  total_customers  retired_pct_of_step
age_step                                                                     
50-54                  174         3651             3825                  4.5
55-59                  652         2384             3036                 21.5
60-64                  225          305              530                 42.5
65-69                  141           53              194                 72.7
70 and over            418           50              468                 89.3

Subscribers by age step and retirement status:
retirement_status  retired  not retired
age_step                               
50-54                   15          357
55-59                   80          245
60-64                   70           89
65-69                   70           22
70 and over            194           27

Yes rate (%) by age step and retirement status (blank where under 100 customers)

In [7]:
# Q3, step 5: observed vs expected yes rate, given each group's age mix
age_edges = [0, 25, 30, 35, 40, 45, 50, 55, 60, 65, 120]
age_labels = ["under 25", "25-29", "30-34", "35-39", "40-44",
              "45-49", "50-54", "55-59", "60-64", "65 and over"]
age_band = pd.cut(clean_df["age"], bins=age_edges, labels=age_labels, right=False)

def observed_vs_expected(column):
    rows = []
    for group in clean_df[column].unique():
        in_group = clean_df[column] == group

        # Yes rate of customers in ALL OTHER groups, within each age band
        other_rates = (subscribed[~in_group]
                       .groupby(age_band[~in_group], observed=True).mean()
                       .reindex(age_labels))
        group_counts = age_band[in_group].value_counts().reindex(age_labels).fillna(0)

        assert not ((group_counts > 0) & other_rates.isna()).any(), f"no comparison customers for {group}"
        expected_subscribers = (group_counts * other_rates).sum()

        customers = in_group.sum()
        observed_subscribers = subscribed[in_group].sum()
        observed_rate = observed_subscribers / customers * 100
        expected_rate = expected_subscribers / customers * 100
        margin = 1.96 * (expected_rate / 100 * (1 - expected_rate / 100) / customers) ** 0.5 * 100

        rows.append({
            column: group,
            "customers": customers,
            "observed_subscribers": observed_subscribers,
            "observed_yes_rate_pct": round(observed_rate, 2),
            "expected_yes_rate_pct": round(expected_rate, 2),
            "difference_points": round(observed_rate - expected_rate, 2),
            "chance_margin_points": round(margin, 2),
            "small_group": customers < 100,
        })

    result = pd.DataFrame(rows).set_index(column)
    result["outside_chance_margin"] = result["difference_points"].abs() > result["chance_margin_points"]
    print(f"{column}: customers {result['customers'].sum()} (should be {len(clean_df)}), "
          f"observed subscribers {result['observed_subscribers'].sum()} (should be {subscribed.sum()})")
    return result.sort_values("difference_points", ascending=False)

job_vs_expected = observed_vs_expected("job")
education_vs_expected = observed_vs_expected("education")

print()
print("Job, compared with the rate expected from its age mix:")
print(job_vs_expected)
print()
print("Education, compared with the rate expected from its age mix:")
print(education_vs_expected)

job: customers 41176 (should be 41176), observed subscribers 4639 (should be 4639)
education: customers 41176 (should be 41176), observed subscribers 4639 (should be 4639)

Job, compared with the rate expected from its age mix:
               customers  observed_subscribers  observed_yes_rate_pct  \
job                                                                     
student              875                   275                  31.43   
unemployed          1014                   144                  14.20   
admin.             10419                  1351                  12.97   
management          2924                   328                  11.22   
retired             1718                   434                  25.26   
technician          6739                   730                  10.83   
self-employed       1421                   149                  10.49   
unknown              330                    37                  11.21   
housemaid           1060                  